In [ ]:
from pathlib import Path
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "project.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside this repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project import input_path, output_path


In [ ]:
import numpy as np
import pandas as pd
import itertools
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import MultipleLocator
from scipy.stats.mstats import zscore
import matplotlib.gridspec as gridspec
from tqdm import tqdm

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.linewidth'] = 0.8  # Thin spines
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

In [ ]:
from matplotlib.lines import Line2D
import matplotlib.cm as cm

# Load

In [ ]:
df = pd.read_csv(input_path('nda_govt_patents.csv'),usecols=['patent_id'])
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
patents_inst = pd.read_parquet(input_path('patent_inventor.parquet'))
patents_inst = patents_inst[patents_inst['patent_id'].isin(df['patent_id'].values)].reset_index(drop=True)

inst_info = pd.read_parquet(input_path('location.parquet'))
inst_count = patents_inst.groupby('location_id')['patent_id'].nunique().reset_index()
inst_count.columns = ['location_id','N_patents']
inst_count = inst_count.sort_values('N_patents',ascending=False).reset_index(drop=True)

for col in ['disambig_country', 'disambig_city', 'latitude', 'longitude','disambig_state']:
    inst_count[col] = inst_count['location_id'].map(inst_info.set_index('location_id')[col])
    patents_inst[col] = patents_inst['location_id'].map(inst_info.set_index('location_id')[col])

In [ ]:
patents_assignee = pd.read_parquet(input_path('patent_assignee.parquet'))
patents_assignee = patents_assignee[patents_assignee['patent_id'].isin(df['patent_id'].values)]
assignee = pd.read_parquet(input_path('assignee.parquet'))
patents_assignee['assignee'] = patents_assignee['assignee_id'].map(assignee.set_index('assignee_id')['disambig_assignee_organization'])
patents_assignee.groupby('assignee').size().reset_index().sort_values(0).tail(7)

In [ ]:
patents_inst['inventor_id'].nunique()

# Bar plot

In [ ]:
import matplotlib.colors as mcolors

def draw_nature_bar_colored(ax, df, cat_col, val_col, panel_label, colormap_name='viridis'):
    # Sort for plotting (largest on top)
    df_sorted = df.sort_values(by=val_col, ascending=True)
    
    # Generate colors based on values
    # Normalize values to 0-1 range for the colormap
    cmap = plt.get_cmap(colormap_name)
    norm = mcolors.Normalize(vmin=df_sorted[val_col].min(), vmax=df_sorted[val_col].max())
    colors = cmap(norm(df_sorted[val_col]))
    bars = ax.barh(df_sorted[cat_col], df_sorted[val_col], color=colors, edgecolor='none', height=0.7)
    
    # Grid
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

    # Dynamic Label Placement and Color
    max_val = df_sorted[val_col].max()
    threshold = max_val * 0.3
    
    for bar, label, color in zip(bars, df_sorted[cat_col], colors):
        width = bar.get_width()
        y_pos = bar.get_y() + bar.get_height() / 2
        
        # Determine text color based on bar color brightness for contrast
        # Convert RGB to HSV and check the Value (brightness) component
        text_color = 'white' if mcolors.rgb_to_hsv(color[:3])[2] < 0.7 else 'black'

        if width > threshold:
            # Inside bar, right aligned
            ax.text(width - (max_val * 0.02), y_pos, label, 
                    ha='right', va='center', color=text_color, 
                    fontsize=9, fontweight='bold', fontfamily='sans-serif')
        else:
            # Outside bar, left aligned, black
            ax.text(max_val * 0.02, y_pos, label, 
                    ha='left', va='center', color='black', 
                    fontsize=9, fontfamily='sans-serif')

    # X-axis label
    ax.set_xlabel('Number of patents', fontsize=10, fontfamily='sans-serif')
    
    # Panel Label
    ax.text(-0.05, 1.02, panel_label, transform=ax.transAxes, 
            fontsize=16, fontweight='bold', va='bottom', ha='right', fontfamily='sans-serif')

In [ ]:
Country_count = patents_inst.groupby('disambig_country')['inventor_id'].nunique().reset_index().sort_values('inventor_id',ascending=False).reset_index(drop=True).head(20)
state_count = patents_inst.groupby('disambig_state')['inventor_id'].nunique().reset_index().sort_values('inventor_id',ascending=False).reset_index(drop=True).head(20)

def draw_nature_bar(ax, df, cat_col, val_col, panel_label, colormap_name='viridis'):
    # Sort for plotting (largest on top)
    df_sorted = df.sort_values(by=val_col, ascending=True)
    n_bars = len(df_sorted)
    # Create bars
    rank_values = np.linspace(0.3, 1.0, n_bars)
    cmap = plt.get_cmap(colormap_name)
    colors = cmap(rank_values)
    bars = ax.barh(df_sorted[cat_col], df_sorted[val_col], color=colors, edgecolor='none', height=0.85)
    
    # Styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False) # Clean look
    ax.set_yticks([]) # Remove default y-ticks
    ax.tick_params(labelsize=11)
    # Grid
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

    # Dynamic Label Placement
    max_val = df_sorted[val_col].max()
    threshold = max_val * 0.6 # Threshold for placing label inside vs outside
    
    for bar, label in zip(bars, df_sorted[cat_col]):
        width = bar.get_width()
        y_pos = bar.get_y() + bar.get_height() / 2 -0.05
        
        if width > threshold:
            # Inside bar, right aligned, white
            ax.text(width - (max_val * 0.02), y_pos, label, 
                    ha='right', va='center', color='white', 
                    fontsize=11, fontweight='bold', fontfamily='sans-serif')
        else:
            # Outside bar (or left aligned), black
            ax.text(width + (max_val * 0.02), y_pos, label, 
                    ha='left', va='center', color='black', 
                    fontsize=11, fontfamily='sans-serif')

    # X-axis label
    ax.set_xlabel('Number of inventors', fontsize=11, fontfamily='sans-serif')
    
    # Panel Label
    ax.text(-0.05, 1.02, panel_label, transform=ax.transAxes, 
            fontsize=16, fontweight='bold', va='bottom', ha='right', fontfamily='sans-serif')

if True: # 4. Generate Figure
    fig = plt.figure(figsize=(9, 4))
    gs = gridspec.GridSpec(1, 2, wspace=0.4)

    ax1 = fig.add_subplot(gs[0, 0])
    draw_nature_bar(ax1, Country_count, 'disambig_country', 'inventor_id', 'b', colormap_name='Greys')
    ax1.set_title('Top Countries', loc='left', fontsize=11, fontweight='bold')

    ax2 = fig.add_subplot(gs[0, 1])
    draw_nature_bar(ax2, state_count, 'disambig_state', 'inventor_id', 'c', colormap_name='Greys')
    ax2.set_title('Top States', loc='left', fontsize=11, fontweight='bold')

    plt.tight_layout()
plt.savefig(output_path('fig_3_cd.svg'))

# World map

In [ ]:
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import country_converter as coco
from shapely.geometry import LineString

In [ ]:
edges = patents_inst.merge(patents_inst, on='patent_id', suffixes=('_source', '_target'))
edges = edges[edges['location_id_source'] < edges['location_id_target']]
network = edges.groupby(['location_id_source', 'location_id_target']).size().reset_index(name='weight')
network = network.merge(inst_count[['location_id', 'longitude', 'latitude']], 
                        left_on='location_id_source', right_on='location_id')
network = network.rename(columns={'longitude': 'lon_source', 'latitude': 'lat_source'})
network = network.merge(inst_count[['location_id', 'longitude', 'latitude']], 
                        left_on='location_id_target', right_on='location_id')
network = network.rename(columns={'longitude': 'lon_target', 'latitude': 'lat_target'})
network = network[['location_id_source', 'location_id_target', 'weight', 
                   'lon_source', 'lat_source', 'lon_target', 'lat_target']]

In [ ]:
network = network[network['weight'] > 1]
network = network.sort_values('weight').reset_index(drop=True)

In [ ]:
def get_lw(weight):
    if max_weight_data == 0:
        return 0
    return (weight / max_weight_data) * 1.2
def get_size(n_patents):
    node_scale_factor = 0.1
    return n_patents * node_scale_factor
max_weight_data = 150 
vmax_color = 100
norm = mcolors.Normalize(vmin=network['weight'].min() if 'weight' in network.keys() else 1, vmax=vmax_color)
cmap = plt.cm.get_cmap('Reds')

In [ ]:
fig = plt.figure(figsize=(15, 10)) 
gs = gridspec.GridSpec(2, 10, figure=fig, hspace=0.1, wspace=0)

# =================================================================
# ## 1. World Map (Main Plot - Bottom Row, Full Width)
# =================================================================
# ax_world spans the entire second row (indices 1) and all 5 columns (0:5)
# =================================================================
# ## 1. US Inset Map (Top Plot)
# =================================================================
ax1 = fig.add_subplot(gs[0, 1:6], projection=ccrs.PlateCarree())
ax1.set_extent([-130, -65, 20, 50])
ax1.set_facecolor("#404040")
ax1.add_feature(cfeature.LAND, color='#070707', zorder=1)
ax1.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5, edgecolor='gray', zorder=2)
ax1.add_feature(cfeature.STATES, linestyle=':', alpha=0.5, edgecolor='gray', zorder=2)

for _, row in network.iterrows():
    if (-130 < row['lon_source'] < -65 and 20 < row['lat_source'] < 50) and \
       (-130 < row['lon_target'] < -65 and 20 < row['lat_target'] < 50):
        ax1.plot([row['lon_source'], row['lon_target']], [row['lat_source'], row['lat_target']], 
                 color=cmap(norm(row['weight'])), linewidth=get_lw(row['weight']), alpha=0.4, 
                 transform=ccrs.Geodetic())

ax1.scatter(inst_count['longitude'], inst_count['latitude'], 
            s=get_size(inst_count['N_patents']), c='white', alpha=0.6, 
            edgecolors='none', transform=ccrs.PlateCarree(), zorder=3)


# =================================================================
# ## 2. World Map (Bottom Plot)
# =================================================================
ax2 = fig.add_subplot(gs[1, 0:10], projection=ccrs.PlateCarree())
ax2.set_extent([-150, 180, -55, 70])
ax2.set_facecolor("#404040")
ax2.add_feature(cfeature.LAND, color='#070707', zorder=1)

for _, row in network.iterrows():
    ax2.plot([row['lon_source'], row['lon_target']], [row['lat_source'], row['lat_target']], 
             color=cmap(norm(row['weight'])), linewidth=get_lw(row['weight']), alpha=0.2, 
             transform=ccrs.Geodetic())

ax2.scatter(inst_count['longitude'], inst_count['latitude'], 
            s=get_size(inst_count['N_patents']), c='white', alpha=0.4, 
            edgecolors='none', transform=ccrs.PlateCarree(), zorder=3)

plt.savefig(output_path('fig_3_map_patent.tiff'), format='tiff')

In [ ]:
### 2. Plotting and Saving Legends and Colorbar (SVG)

# Create a new, small figure for the legends only
fig_legend = plt.figure(figsize=(4, 6))
ax_legend = fig_legend.add_subplot(111)

# Set the background to white/transparent
ax_legend.set_axis_off() 

# --- Legend 1: Node Size (Publications) - Black Text ---
node_legend_vals = [10, 50, 100, 400] 
node_handles = [plt.scatter([], [], s=get_size(val), c='gray', edgecolors='k', alpha=0.8, linewidth=0.5) 
                for val in node_legend_vals]

legend_nodes = ax_legend.legend(node_handles, [str(v) for v in node_legend_vals], loc="upper left", 
                                title="Teams", frameon=False, labelcolor='k', 
                                bbox_to_anchor=(0.0, 1.0))
plt.setp(legend_nodes.get_title(), color='k', fontsize=11)
plt.setp(legend_nodes.get_texts(), fontsize=11)
ax_legend.add_artist(legend_nodes) 

# --- Legend 2: Edge Width (Collaboration Strength) - Black Text ---
edge_legend_vals = [10, 50, 100, 150] 
line_handles = [Line2D([0], [0], color='gray', linewidth=get_lw(val), alpha=0.8) 
                for val in edge_legend_vals]

# Position the edge legend below the node legend
legend_edges = ax_legend.legend(line_handles, [f"{int(v)}" for v in edge_legend_vals], 
                                loc="upper left", title="Collab. Strength", frameon=False, 
                                labelcolor='k', bbox_to_anchor=(0.0, 0.6))
plt.setp(legend_edges.get_title(), color='k', fontsize=11)
plt.setp(legend_edges.get_texts(), fontsize=11)

# --- Colorbar ---
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([]) 
tick_locs = [0, 10, 50, 100]

# Add a fake axis for the colorbar to attach to, placed on the side
cbar_ax = fig_legend.add_axes([0.5, 0.4, 0.05, 0.45]) # [left, bottom, width, height]

cbar = plt.colorbar(sm, cax=cbar_ax, location='right', orientation='vertical', ticks=tick_locs)
cbar.set_label('Collab. Strength', color='k', fontsize=11)
cbar.ax.yaxis.set_tick_params(color='k', labelcolor='k', labelsize=11)
cbar.ax.set_yticklabels([f"{t}" for t in tick_locs])
cbar.outline.set_edgecolor('k') # Black outline for Nature style

# Save Plot 2 as SVG (Legends and Colorbar)
plt.savefig(output_path('fig_3_legends_patent.svg'), format='svg', transparent=True)